In [1]:
FAISS_PATH='../vectorstore/faiss_index'

In [2]:
from langchain_ollama import ChatOllama

llm = ChatOllama(
    model = "llama3.2:3b",
    temperature=0.0
)
response = llm.invoke("Hola, funcionas en local?")
print(response.content)

¡Hola! Me alegra que hayas intentado contactarme. Sin embargo, como soy una inteligencia artificial, no tengo una ubicación física específica y puedo acceder a internet desde cualquier lugar.

Puedo proporcionarte información y responder a tus preguntas de manera remota, siempre y cuando tengas acceso a una conexión a Internet. ¿En qué puedo ayudarte hoy?


In [5]:
from langchain_community.document_loaders import UnstructuredMarkdownLoader
from langchain_community.document_loaders import DirectoryLoader
from langchain_community.document_loaders import JSONLoader

# cargamos la knowledge
folder_loader = DirectoryLoader(
    "../knowledge/production_simulation",
    glob="**/*.md",
    loader_cls=UnstructuredMarkdownLoader
)
docs_md = folder_loader.load()

# cargamos los json
def json_metadata(record: dict, metadata: dict):
    metadata["traceId"] = record.get("traceId", "NONE")
    metadata["service"] = record.get("service", "NONE")
    metadata["level"] = record.get("level", "NONE")

    return metadata

json_arguments = {
    "jq_schema": '.[] | select(has("error_message"))',
    "content_key": "error_message",
    "metadata_func": json_metadata
}

json_loader = DirectoryLoader(
    "../datasets",
    glob="**/*.json",
    loader_cls=JSONLoader,
    loader_kwargs=json_arguments
)
docs_json = json_loader.load()

print(f"Archivos .md cargados: {len(docs_md)}")
print(f"Archivos JSON cargados: {len(docs_json)}")

Archivos .md cargados: 5
Archivos JSON cargados: 2279


In [6]:
docs = docs_md + docs_json

# ahora con los datos completos los pasamos por el textsplitter y preparamos el vectorstore
from langchain_text_splitters import RecursiveCharacterTextSplitter, MarkdownHeaderTextSplitter
from langchain_ollama import OllamaEmbeddings
from langchain_community.vectorstores import FAISS

headers_to_split_on = [
    ("#", "titulo"),
    ("##", "seccion"),
    ("###", "subseccion"),
]

md_splitter = MarkdownHeaderTextSplitter(headers_to_split_on=headers_to_split_on)

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=150,
    length_function=len,
)

embedding = OllamaEmbeddings(
    model = "nomic-embed-text"
)

chunks = text_splitter.split_documents(docs)

vectorstore = FAISS.from_documents(chunks, embedding)

vectorstore.save_local(FAISS_PATH)

retriever = vectorstore.as_retriever()

ResponseError: Post "http://127.0.0.1:53323/tokenize": EOF (status code: 400)

In [ ]:
from langchain_core.runnables import RunnablePassthrough
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from operator import itemgetter
from langchain_core.output_parsers import StrOutputParser

template = """
    Eres un Ingeniero SRE (Site Reliability Engineer) y Experto Forense de Nivel 3. 
    Tu especialidad es diagnosticar fallos en cascada en una arquitectura de microservicios Spring Boot (api-pedidos, api-inventario, api-autenticacion).

    Has recibido una alerta de nuestro modelo de Machine Learning (Isolation Forest) indicando que el siguiente bloque de logs es una ANOMALÍA CRÍTICA.

    REGLAS ESTRICTAS:
    1. NO inventes información. Utiliza ÚNICAMENTE la documentación técnica y los runbooks proporcionados en el apartado <contexto>.
    2. Si el <contexto> no contiene la respuesta, di explícitamente: "No hay información suficiente en los manuales para determinar la causa raíz."
    3. Debes diferenciar el "paciente cero" (causa raíz) de las víctimas (errores en cascada).

    <contexto>
    {context}
    </contexto>

    FORMATO DE RESPUESTA OBLIGATORIO:
    Responde siempre usando esta estructura en Markdown:

    🚨 **Análisis de Causa Raíz (RCA)**
    * **Microservicio Origen:** [Nombre del servicio que falló primero]
    * **Excepción Principal:** [Tipo de error, ej. NullPointerException, Timeout]
    * **Diagnóstico:** [Explicación técnica de 2 o 3 líneas de por qué ocurrió según el contexto]

    🛠️ **Plan de Mitigación (Runbook)**
    1. [Paso 1 para solucionarlo]
    2. [Paso 2 para solucionarlo]
"""
prompt = ChatPromptTemplate.from_messages([
    ("system", template),
    MessagesPlaceholder(variable_name="history"),
    ("human:" "Analiza este log anomalo y dime qué ha pasado: \n\n{input}")
])

def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

rag_chain = (
    RunnablePassthrough.assign(
        context=itemgetter("input") | retriever | format_docs
    )
    | prompt
    | llm
    | StrOutputParser()
)

In [ ]:
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_community.chat_message_histories import ChatMessageHistory

# almacenamos el historial segun el usuario
chat_history = ChatMessageHistory()

def obtener_historial_por_session_id(session_id: str):
  return chat_history

RAG = RunnableWithMessageHistory(
  rag_chain,
  obtener_historial_por_session_id,
  input_messages_key="input",
  history_messages_key="history"
)

/opt/homebrew/Caskroom/miniforge/base/envs/rca/lib/python3.11/site-packages/IPython/core/interactiveshell.py:3748: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


In [ ]:
test_log = """
TRACE_ID: 6a721f115975b8d2d0fd94c37ef6598a

[08:00:00] SERVICIO: api-inventario | EVENTO: SERVER_ERROR | DURACION: 30106ms
EXCEPCION: CannotCreateTransactionException
MENSAJE: could not open jpa entitymanager for transaction

[08:00:05] SERVICIO: api-pedidos | EVENTO: UNHANDLED_ERROR | DURACION: 5185ms
EXCEPCION: ResourceAccessException
MENSAJE: i o error on post request for http api inventario - read timed out
}
"""

# Ejecutamos el bot
respuesta = RAG.invoke(
    {"input": test_log},
    config={"configurable": {"session_id": "prueba_test_01"}}
)
print(respuesta)

Basado en este log anomalo, parece que ha habido un problema con la conexión a la base de datos cuando se llama al método `retirarStockeDeInventario` del servicio `api-inventario`. El error es un `CannotCreateTransactionException`, lo que indica que no se pudo crear una transacción para el manejo de excepciones.

El log muestra dos eventos consecutivos:

1. Un error 500.0 en la API `api-inventario` debido a una llamada fallida al método `retirarStockeDeInventario`. El mensaje del error es "could not open jpa entitymanager for transaction".
2. Un error 500.0 en la API `api-pedidos` debido a una llamada fallida al servicio `PedidoService`. El mensaje del error es "i o error on post request for http api inventario - read timed out".

El hecho de que ambos servicios tengan un error 500.0 y mensajes de error similares sugiere que hay un problema con la conexión a la base de datos que se ha propagado a ambos servicios.

En particular, el mensaje del error "could not open jpa entitymanager fo